Завдання 1

Встановлення та імпорт бібліотек

In [1]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.6 MB/s  0:00:00


In [2]:
import pandas as pd
import numpy as np
import os

from surprise import Dataset, Reader, SVD, KNNBasic
from surprise.model_selection import GridSearchCV
from surprise import accuracy

Завантаження датасету MovieLens 100K

In [3]:
data = Dataset.load_builtin('ml-100k')

print("Датасет успішно завантажено!")
print("Кількість записів:", len(data.raw_ratings))

Dataset ml-100k could not be found. Do you want to download it? [Y/n] y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Датасет успішно завантажено!
Кількість записів: 100000


Виведення перших 5 рядків

In [4]:
df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print("Перші 5 рядків датасету:")
display(df.head())

Перші 5 рядків датасету:


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


Підбір параметрів першого алгоритму SVD

In [5]:
param_grid_svd = {
    'n_factors': [50, 100],
    'n_epochs': [20, 30],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.02, 0.05]
}

gs_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['mae'],
    cv=3,
    n_jobs=-1
)

gs_svd.fit(data)

print("Найкращі параметри SVD:")
print(gs_svd.best_params['mae'])

print("Найкраще значення MAE:")
print(gs_svd.best_score['mae'])

Найкращі параметри SVD:
{'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.01, 'reg_all': 0.05}
Найкраще значення MAE:
0.7330564261330474


Підбір параметрів другого алгоритму KNNBasic

In [7]:
param_grid_knn = {
    'k': [20, 40, 60],
    'min_k': [1, 3],
    'sim_options': {
        'name': ['cosine', 'msd'],
        'user_based': [True, False]
    }
}

gs_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['mae'],
    cv=3,
    n_jobs=-1
)

gs_knn.fit(data)

print("Найкращі параметри KNNBasic:")
print(gs_knn.best_params['mae'])

print("Найкраще значення MAE:")
print(gs_knn.best_score['mae'])

Найкращі параметри KNNBasic:
{'k': 20, 'min_k': 1, 'sim_options': {'name': 'msd', 'user_based': True}}
Найкраще значення MAE:
0.7795137655752393


Порівняння алгоритмів та вибір найкращого

In [8]:
mae_svd = gs_svd.best_score['mae']
mae_knn = gs_knn.best_score['mae']

results = pd.DataFrame({
    'Алгоритм': ['SVD', 'KNNBasic'],
    'MAE': [mae_svd, mae_knn]
})

print("Порівняння алгоритмів:")
display(results)

if mae_svd < mae_knn:
    best_algo = gs_svd.best_estimator['mae']
    best_name = 'SVD'
else:
    best_algo = gs_knn.best_estimator['mae']
    best_name = 'KNNBasic'

print("Найкращий алгоритм:", best_name)
print("Значення MAE:", min(mae_svd, mae_knn))

Порівняння алгоритмів:


,Алгоритм,MAE
0,SVD,0.733056
1,KNNBasic,0.779514


Найкращий алгоритм: SVD
Значення MAE: 0.7330564261330474


Навчання найкращої моделі

In [9]:
trainset = data.build_full_trainset()
best_algo.fit(trainset)

print("Модель успішно навчена!")

Модель успішно навчена!


Отримання назв фільмів

In [12]:
file_path = os.path.join(
    os.path.expanduser("~"),
    ".surprise_data",
    "ml-100k",
    "ml-100k",
    "u.item"
)

movies = pd.read_csv(
    file_path,
    sep='|',
    encoding='latin-1',
    header=None,
    usecols=[0, 1],
    names=['item_id', 'title']
)

movies['item_id'] = movies['item_id'].astype(str)

print("Назви фільмів успішно завантажено!")
print("Кількість фільмів:", len(movies))

display(movies.head())

Назви фільмів успішно завантажено!
Кількість фільмів: 1682


,item_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


Рекомендація 10 фільмів конкретному користувачу

In [13]:
user_id = '1'

watched_movies = set(
    df[df['user_id'] == user_id]['item_id'].astype(str)
)

all_movies = movies['item_id'].tolist()

unwatched_movies = [
    movie for movie in all_movies
    if movie not in watched_movies
]

predictions = []

for movie_id in unwatched_movies:
    prediction = best_algo.predict(user_id, movie_id)

    predictions.append({
        'item_id': str(movie_id),
        'predicted_rating': prediction.est
    })

recommendations = pd.DataFrame(predictions)

recommendations = recommendations.merge(
    movies,
    on='item_id',
    how='inner'
)

recommendations = recommendations.sort_values(
    by='predicted_rating',
    ascending=False
).head(10)

print("10 рекомендованих фільмів для користувача", user_id)

display(
    recommendations[['title', 'predicted_rating']]
)

10 рекомендованих фільмів для користувача 1


,title,predicted_rating
238,Lawrence of Arabia (1962),5.000000
135,"Close Shave, A (1995)",5.000000
240,"Third Man, The (1949)",4.845950
210,Casablanca (1942),4.836255
12,Secrets & Lies (1996),4.824483
201,Dr. Strangelove or: How I Learned to Stop Worr...,4.821807
646,"City of Lost Children, The (1995)",4.755385
257,"Man Who Would Be King, The (1975)",4.728051
239,Wings of Desire (1987),4.721698
1176,Pather Panchali (1955),4.716328


Завдання 2

Підготовка даних

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity

train_data, test_data = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

train_matrix = train_data.pivot_table(
    index='user_id',
    columns='item_id',
    values='rating'
)

train_matrix_filled = train_matrix.fillna(0)

print("Розмір матриці оцінок:", train_matrix.shape)
display(train_matrix.head())

Розмір матриці оцінок: (943, 1653)


item_id,1,10,100,1000,1001,1002,1003,1004,1005,1006,...,990,991,992,993,994,995,996,997,998,999
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,4.0,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
101,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN


Обчислення схожості користувачів

In [15]:
similarity_matrix = cosine_similarity(train_matrix_filled)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=train_matrix.index,
    columns=train_matrix.index
)

print("Матриця схожості користувачів:")
display(similarity_df.iloc[:5, :5])

Матриця схожості користувачів:


user_id,1,10,100,101,102
user_id,,,,,
1,1.000000,0.281396,0.043192,0.115399,0.320418
10,0.281396,1.000000,0.055794,0.020950,0.235101
100,0.043192,0.055794,1.000000,0.016326,0.113536
101,0.115399,0.020950,0.016326,1.000000,0.158659
102,0.320418,0.235101,0.113536,0.158659,1.000000


Реалізація власного алгоритму прогнозування

In [16]:
user_means = train_matrix.mean(axis=1)

global_mean = train_data['rating'].mean()


def predict_rating(user_id, item_id):

    if user_id not in train_matrix.index:
        return global_mean

    user_mean = user_means[user_id]

    if item_id not in train_matrix.columns:
        return user_mean

    item_ratings = train_matrix[item_id].dropna()

    item_ratings = item_ratings.drop(
        index=user_id,
        errors='ignore'
    )

    if item_ratings.empty:
        return user_mean

    similarities = similarity_df.loc[
        user_id, item_ratings.index
    ]

    deviations = item_ratings - user_means.loc[item_ratings.index]

    denominator = np.abs(similarities).sum()

    if denominator == 0:
        return user_mean

    prediction = user_mean + (
        np.dot(similarities, deviations) / denominator
    )

    return np.clip(prediction, 1, 5)


print("Власний алгоритм рекомендацій успішно створено!")

Власний алгоритм рекомендацій успішно створено!


Оцінювання власної рекомендаційної системи

In [17]:
predicted_ratings = []

for _, row in test_data.iterrows():
    predicted_ratings.append(
        predict_rating(row['user_id'], row['item_id'])
    )

mae_own = mean_absolute_error(
    test_data['rating'],
    predicted_ratings
)

print("Оцінювання власної рекомендаційної системи:")
print("MAE:", round(mae_own, 4))

Оцінювання власної рекомендаційної системи:
MAE: 0.7471


Отримання рекомендацій для користувача

In [18]:
user_id = '1'

watched = set(
    train_data[train_data['user_id'] == user_id]['item_id']
)

all_items = train_matrix.columns.tolist()

unwatched = [
    item for item in all_items
    if item not in watched
]

own_recommendations = []

for item_id in unwatched:
    rating = predict_rating(user_id, item_id)

    own_recommendations.append({
        'item_id': item_id,
        'predicted_rating': rating
    })

own_recommendations = pd.DataFrame(own_recommendations)

own_recommendations = own_recommendations.merge(
    movies,
    on='item_id'
)

own_recommendations = own_recommendations.sort_values(
    by='predicted_rating',
    ascending=False
).head(10)

print("10 рекомендацій власної системи для користувача", user_id)

display(
    own_recommendations[['title', 'predicted_rating']]
)

10 рекомендацій власної системи для користувача 1


,title,predicted_rating
504,Santa with Muscles (1996),5.000000
466,"Boys, Les (1997)",5.000000
631,Nico Icon (1995),5.000000
640,Some Mother's Son (1996),5.000000
470,"Saint of Fort Washington, The (1993)",5.000000
312,Delta of Venus (1994),5.000000
371,Mina Tannenbaum (1994),4.998453
649,Entertaining Angels: The Dorothy Day Story (1996),4.973214
554,Destiny Turns on the Radio (1995),4.895112
434,Sliding Doors (1998),4.865421


Порівняння трьох рекомендаційних систем

In [19]:
comparison = pd.DataFrame({
    'Рекомендаційна система': [
        'SVD',
        'KNNBasic',
        'Власна система (User-Based CF)'
    ],
    'MAE': [
        mae_svd,
        mae_knn,
        mae_own
    ]
})

print("Порівняння рекомендаційних систем:")
display(comparison.sort_values(by='MAE'))

best_system = comparison.loc[
    comparison['MAE'].idxmin()
]

print("Найкраща система:", best_system['Рекомендаційна система'])
print("Найменше значення MAE:", round(best_system['MAE'], 4))

Порівняння рекомендаційних систем:


,Рекомендаційна система,MAE
0,SVD,0.733056
2,Власна система (User-Based CF),0.747069
1,KNNBasic,0.779514


Найкраща система: SVD
Найменше значення MAE: 0.7331


Висновок: Під час виконання лабораторної роботи було досліджено
принципи побудови рекомендаційних систем. У першому завданні було завантажено та проаналізовано датасет MovieLens 100K за допомогою бібліотеки Surprise. Реалізовано два алгоритми рекомендацій: SVD та KNNBasic. За допомогою крос-валідації GridSearchCV було підібрано
оптимальні параметри для обох алгоритмів. Для порівняння моделей використано метрику MAE. На основі отриманих результатів визначено найкращий алгоритм та сформовано рекомендації з 10 фільмів
для конкретного користувача. У другому завданні було розроблено власну рекомендаційну систему на основі колаборативної фільтрації за користувачами. Для визначення схожості використовувалася косинусна схожість. Якість системи оцінено за допомогою MAE та отримано
персональні рекомендації для користувача. У результаті роботи було отримано практичні навички завантаження та обробки даних, реалізації рекомендаційних алгоритмів, використання крос-валідації та оцінювання
точності рекомендаційних систем.